# 04 – Inventory Optimization

**Owner:** Bhumika KN. This notebook demonstrates the inventory module with synthetic data. Replace it with the approved project dataset and forecast output before reporting final results.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.inventory.safety_stock import safety_stock_from_history
from src.inventory.reorder_point import calculate_reorder_point, replenishment_recommendation
from src.inventory.inventory_simulation import simulate_inventory, compare_inventory_scenarios

## 1. Load illustrative demand history

In [ ]:
data_path = ROOT / 'data' / 'dummy_daily_demand.csv'
df = pd.read_csv(data_path, parse_dates=['date'])
df.head()

In [ ]:
sku = 'P001'
sku_df = df[df['sku'] == sku].sort_values('date')
daily_demand = sku_df['actual_demand'].tolist()
avg_daily_demand = float(sku_df['actual_demand'].mean())
lead_time_days = 5
service_level = 0.95

print('SKU:', sku)
print('Average daily demand:', round(avg_daily_demand, 2))
print('Historical standard deviation:', round(sku_df['actual_demand'].std(), 2))

## 2. Calculate safety stock and reorder point

In [ ]:
safety_stock = safety_stock_from_history(daily_demand, lead_time_days, service_level)
reorder_point = calculate_reorder_point(avg_daily_demand, lead_time_days, safety_stock)
print('Safety stock:', safety_stock)
print('Reorder point:', reorder_point)

## 3. Replenishment recommendation

In [ ]:
recommendation = replenishment_recommendation(
    inventory_on_hand=400, reorder_point=reorder_point,
    order_quantity=500, on_order=0, backorders=0
)
recommendation

## 4. Simulate inventory using a held-out demand sequence

In [ ]:
# Synthetic demand sequence for demonstration. This is not model forecast output.
simulation_demand = [100, 105, 98, 110, 102, 108, 99, 155, 103, 107, 101, 112, 97, 109, 104]
results, summary = simulate_inventory(
    simulation_demand, initial_inventory=600,
    reorder_point=reorder_point, order_quantity=500,
    lead_time_days=lead_time_days, safety_stock=safety_stock
)
display(results)
summary

In [ ]:
ax = results.plot(x='date', y=['closing_inventory', 'on_order'], marker='o', figsize=(10, 4), title='Inventory position over simulation')
ax.set_ylabel('Units')
plt.tight_layout()
plt.show()

## 5. Compare what-if scenarios

In [ ]:
scenarios = {
    'Normal lead time': {'initial_inventory': 600, 'reorder_point': reorder_point, 'order_quantity': 500, 'lead_time_days': 5},
    'Longer lead time': {'initial_inventory': 600, 'reorder_point': reorder_point, 'order_quantity': 500, 'lead_time_days': 8},
    'Lower starting stock': {'initial_inventory': 350, 'reorder_point': reorder_point, 'order_quantity': 500, 'lead_time_days': 5},
}
scenario_results = compare_inventory_scenarios(simulation_demand, scenarios)
display(scenario_results)

## 6. Interpretation and limitations

Compare fill rate, stockout-day risk, lost sales and average inventory across scenarios. The simulation currently assumes fixed order quantities, a known fixed lead time and lost sales (unfulfilled demand is not backordered). Synthetic outputs are for software verification only. Use actual demand and agreed forecast inputs for the final review.